# Сказочный бот — финальная модель (ноутбук 1)

Этот ноутбук пишет **одну главу сказки** по выбору ребёнка. Модель: ruGPT3 medium, дообученная методом LoRA на русских сказках (адаптер v4, этап 5 диплома).

**Как пользоваться (3 шага):**
1. Запустите ячейки сверху вниз до раздела «Запуск»: они только определяют функции.
2. В последней ячейке запустите `run_demo()`.
3. Отвечайте на вопросы: герои, место, цель, настроение, выбор ребёнка. Через минуту появится глава и отчёт, какие герои названы в тексте.

**Какие файлы должны лежать рядом с ноутбуком (в одной папке):**

| Файл или папка | Что это |
|---|---|
| `adapter_v4/` | обученный адаптер: `adapter_config.json` и `adapter_model.safetensors` |
| `test_sample.jsonl` | 30 примеров из тестовой части данных (модель их при обучении не видела) |
| `history.json` | история обучения для графика (необязательно) |

Базовая модель `ai-forever/rugpt3medium_based_on_gpt2` скачивается из интернета при первом запуске (около 1,7 ГБ). Нужна видеокарта (Colab: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU). На процессоре тоже работает, но одна глава пишется несколько минут.

Обучение и все эксперименты показаны во втором ноутбуке (`02_experiments.ipynb`).

## 1. Установка библиотек

In [ ]:
!pip install -q peft pymorphy3 pymorphy3-dicts-ru
!pip uninstall -y -q torchao   # в Colab старая версия torchao ломает peft


## 2. Папка с файлами

Если ноутбук запущен в Google Colab, укажите папку на Google Диске, где лежат `adapter_v4/`, `test_sample.jsonl` и `history.json`. На своём компьютере ячейка ничего не меняет: файлы должны лежать рядом с ноутбуком.

In [ ]:
import os
import sys

WORK_DIR = "/content/drive/MyDrive/Fairytale_AI_constructor/stage5"   # поменяйте на свою папку

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(WORK_DIR):
        os.chdir(WORK_DIR)
    else:
        print("Папка не найдена:", WORK_DIR, "— поправьте WORK_DIR или загрузите файлы в /content")
print("Рабочая папка:", os.getcwd())
print("Файлы:", sorted(os.listdir(".")))

## 3. Настройки по умолчанию и импорт

In [ ]:
import json
import os
import re
import sys
from functools import lru_cache

import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model

BASE_MODEL = "ai-forever/rugpt3medium_based_on_gpt2"
ADAPTER_DIR = "adapter_v4"            # папка с обученным адаптером
SAMPLE_PATH = "test_sample.jsonl"     # примеры из тестовой части
HISTORY_PATH = "history.json"         # история обучения (для графика)
MOODS = ("calm", "adventure", "scary")   # спокойная / приключение / страшноватая (мягко)
CONTEXT = 2048                        # окно модели в токенах

## 4. Функции

Каждая функция делает одну вещь и использует только свои аргументы (пути и настройки передаются параметрами со значениями по умолчанию). Главная функция `run_demo` собирает их в цепочку: загрузка → очистка → предобработка → loader → модель → генерация → постобработка → проверка героев.

### 4.1. Загрузка данных

In [ ]:
def load_examples(path=SAMPLE_PATH):
    # Читает файл jsonl: каждая строка это один пример (подсказка и, возможно, настоящая глава)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл {path} не найден. Положите его рядом с ноутбуком.")
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def upload_examples():
    # Только для Colab: открывает окно загрузки файла jsonl и читает его
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    with open(name, "wb") as f:
        f.write(uploaded[name])
    return load_examples(name)

### 4.2. Очистка данных

In [ ]:
def parse_prompt(prompt):
    # Разбирает готовую подсказку из датасета в поля запроса
    fields = {}
    for line in prompt.splitlines():
        for key, name in [("Настроение:", "mood"), ("Герои:", "heroes"), ("Место:", "place"),
                          ("Цель:", "goal"), ("Мораль:", "moral"), ("Выбор ребёнка:", "choice")]:
            if line.startswith(key):
                fields[name] = line[len(key):].strip()
        m = re.match(r"Глава (\d+) из (\d+):", line)
        if m:
            fields["chapter"], fields["n_chapters"] = int(m.group(1)), int(m.group(2))
    return fields


def clean_request(raw, moods=MOODS):
    # Очищает запрос: убирает лишние пробелы, проверяет настроение, делает список героев
    if "prompt" in raw:
        raw = {**parse_prompt(raw["prompt"]), **{k: v for k, v in raw.items() if k != "prompt"}}
    squash = lambda s: re.sub(r"\s+", " ", str(s or "")).strip()
    heroes = raw.get("heroes", [])
    if isinstance(heroes, str):
        heroes = heroes.split(",")
    heroes = [squash(h) for h in heroes if squash(h)]
    heroes = list(dict.fromkeys(heroes))[:5]           # без повторов, не больше пяти героев
    if not heroes:
        raise ValueError("Нужен хотя бы один герой.")
    mood = squash(raw.get("mood", "calm")).lower()
    if mood not in moods:
        mood = "calm"
    n_chapters = max(1, min(int(raw.get("n_chapters", 3) or 3), 8))
    chapter = max(1, min(int(raw.get("chapter", 1) or 1), n_chapters))
    return {
        "mood": mood,
        "heroes": heroes,
        "place": squash(raw.get("place")) or "волшебный лес",
        "goal": squash(raw.get("goal")) or "найти друзей",
        "moral": squash(raw.get("moral")) or "доброта и смелость помогают в трудную минуту",
        "choice": squash(raw.get("choice")) or "отправиться в путь",
        "chapter": chapter,
        "n_chapters": n_chapters,
    }

### 4.3. Предобработка: запрос превращается в подсказку для модели

In [ ]:
def preprocess_request(clean):
    # Собирает подсказку в том же формате, на котором модель обучалась
    return (
        f"Настроение: {clean['mood']}\n"
        f"Герои: {', '.join(clean['heroes'])}\n"
        f"Место: {clean['place']}\n"
        f"Цель: {clean['goal']}\n"
        f"Мораль: {clean['moral']}\n"
        "Предыдущие главы: нет\n"
        f"Выбор ребёнка: {clean['choice']}\n"
        f"Глава {clean['chapter']} из {clean['n_chapters']}:\n"
    )

### 4.4. Модель и токенизатор

In [ ]:
def get_model(base_name=BASE_MODEL, adapter_dir=ADAPTER_DIR, preload=True, lora_r=16):
    # preload=True: загружает базовую модель и обученный адаптер из папки adapter_dir.
    # preload=False: создаёт новую необученную модель с адаптером LoRA (её нужно обучать, см. ноутбук 2).
    tok = AutoTokenizer.from_pretrained(base_name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(base_name)
    if preload:
        if not os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
            raise FileNotFoundError(f"В папке {adapter_dir} нет адаптера (adapter_config.json). "
                                    "Положите папку adapter_v4 рядом с ноутбуком.")
        model = PeftModel.from_pretrained(model, adapter_dir).merge_and_unload()
    else:
        cfg = LoraConfig(r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05,
                         target_modules=["c_attn", "c_proj", "c_fc"], fan_in_fan_out=True,
                         bias="none", task_type="CAUSAL_LM")
        model = get_peft_model(model, cfg)
        print("Создана необученная модель с адаптером LoRA: перед использованием обучите её (ноутбук 2).")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    if device == "cuda":
        model = model.half()
    return model.eval(), tok

### 4.5. Loader: подсказки превращаются в пачки токенов

In [ ]:
def make_loader(prompts, tok, batch_size=1, max_prompt_tokens=CONTEXT - 450):
    # Превращает подсказки в список пачек токенов (padding слева, как нужно для генерации)
    for p in prompts:
        if len(tok.encode(p, add_special_tokens=False)) > max_prompt_tokens:
            raise ValueError("Подсказка слишком длинная для окна модели: сократите текст.")
    tok.padding_side = "left"
    return [tok(prompts[s:s + batch_size], return_tensors="pt", padding=True, add_special_tokens=False)
            for s in range(0, len(prompts), batch_size)]

### 4.6. Генерация главы

In [ ]:
def generate_chapters(model, tok, loader, max_new_tokens=400, top_p=0.92, temperature=0.8,
                      no_repeat_ngram_size=4, seed=42):
    # Пишет главу для каждой подсказки. Запрет повторов 4-грамм вместо штрафа:
    # штраф repetition_penalty мешает модели называть героев (см. эксперимент 1).
    device = next(model.parameters()).device
    torch.manual_seed(seed)
    texts = []
    for enc in loader:
        enc = enc.to(device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, do_sample=True, top_p=top_p,
                temperature=temperature, no_repeat_ngram_size=no_repeat_ngram_size,
                eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id,
            )
        for row in out:
            texts.append(tok.decode(row[enc["input_ids"].shape[1]:], skip_special_tokens=True))
    return texts

### 4.7. Постобработка текста

In [ ]:
def postprocess_text(text):
    # Убирает служебные символы и лишние пробелы; если глава оборвалась на полуслове, обрезает до конца предложения
    text = text.replace("</s>", "").strip()
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    if text and text[-1] not in '.!?…»"':
        ends = [m.end() for m in re.finditer(r'[.!?…]["»]?', text)]
        if ends and ends[-1] > len(text) * 0.5:
            text = text[:ends[-1]]
    return text.strip()

### 4.8. Проверка: названы ли герои из запроса

Герой считается названным, если в тексте есть все значимые слова его названия в любой падежной форме (слова приводятся к начальной форме). Это та же проверка, что и в метриках ноутбука 2.

In [ ]:
GENERIC = {"мальчик", "девочка", "мать", "мама", "отец", "папа", "сын", "дочь", "брат", "сестра",
           "старик", "старуха", "дед", "баба", "царь", "король", "принц", "принцесса", "царевна",
           "молодой", "маленький", "старый", "большой", "добрый", "злой", "бедный"}


@lru_cache(maxsize=None)
def get_morph():
    # Морфологический анализатор (создаётся один раз)
    import pymorphy3
    return pymorphy3.MorphAnalyzer()


def lemmas(text, morph=None):
    # Слова текста в начальной форме
    morph = morph or get_morph()
    return [morph.parse(w)[0].normal_form.replace("ё", "е") for w in re.findall(r"[а-яё]+", text.lower())]


def check_heroes(heroes, text, morph=None, generic=GENERIC):
    # Возвращает два списка: какие герои названы в тексте и какие нет
    present = set(lemmas(text, morph))
    named, missing = [], []
    for hero in heroes:
        keys = [k for k in lemmas(hero, morph) if len(k) >= 2]
        keys = [k for k in keys if k not in generic] or keys
        (named if keys and all(k in present for k in keys) else missing).append(hero)
    return named, missing

### 4.9. График обучения и проверка качества на тестовых данных

In [ ]:
def plot_history(history_path=HISTORY_PATH):
    # Рисует кривые обучения: ошибку на обучающих и проверочных данных и перплексию
    if not os.path.exists(history_path):
        print(f"Файл {history_path} не найден, график пропущен.")
        return
    with open(history_path, encoding="utf-8") as f:
        h = json.load(f)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["step"], h["train_loss"], alpha=0.35, label="train (по шагам)")
    axes[0].plot(h["val_step"], h["val_loss"], marker="o", label="val")
    axes[0].set_title("Ошибка при обучении")
    axes[0].set_xlabel("шаг")
    axes[0].set_ylabel("loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(h["val_step"], [2.718281828 ** v for v in h["val_loss"]], marker="o", color="#eb6834")
    axes[1].set_title("Перплексия на val (шаг 0 — базовая модель)")
    axes[1].set_xlabel("шаг")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_sample(model, tok, examples, n=10, max_new_tokens=350):
    # Пишет главы по n подсказкам из тестовых примеров и считает долю названных героев;
    # то же считает для настоящих глав из датасета (для сравнения)
    n = min(n, len(examples))
    cleans = [clean_request(ex) for ex in examples[:n]]
    prompts = [preprocess_request(c) for c in cleans]
    texts = [postprocess_text(t) for t in generate_chapters(model, tok, make_loader(prompts, tok, batch_size=5),
                                                           max_new_tokens=max_new_tokens)]
    def share(pairs):
        vals = []
        for c, t in pairs:
            named, _ = check_heroes(c["heroes"], t)
            vals.append(len(named) / len(c["heroes"]))
        return sum(vals) / len(vals)
    model_cov = share(zip(cleans, texts))
    ref = [(c, ex["completion"].replace("</s>", "")) for c, ex in zip(cleans, examples[:n]) if "completion" in ex]
    ref_cov = share(ref) if ref else float("nan")
    return {"n": n, "model": model_cov, "reference": ref_cov}

### 4.10. Вывод результата и ввод запроса

In [ ]:
def show_result(clean, text, named, missing):
    # Печатает главу и понятный отчёт: какие герои названы
    print("=" * 70)
    print(f"Глава {clean['chapter']} из {clean['n_chapters']} | настроение: {clean['mood']}")
    print(f"Место: {clean['place']} | Цель: {clean['goal']}")
    print(f"Выбор ребёнка: {clean['choice']}")
    print("=" * 70)
    print(text)
    print("=" * 70)
    print("Герои названы в тексте:", ", ".join(named) if named else "ни один")
    if missing:
        print("Не названы:", ", ".join(missing), "(можно запустить ещё раз: текст каждый раз немного другой)")


def ask_request():
    # Спрашивает у пользователя данные для главы (пустой ответ означает значение по умолчанию)
    print("Введите данные для сказки (пустой ответ: значение по умолчанию).")
    heroes = input("Герои через запятую, например: Маша, Серый волк: ")
    return {
        "heroes": heroes,
        "place": input("Место действия, например: тёмный лес: "),
        "goal": input("Цель героев, например: найти потерянную звезду: "),
        "mood": input("Настроение (calm — спокойная, adventure — приключение, scary — страшноватая): "),
        "choice": input("Что выбирает ребёнок, например: пойти по тропинке: "),
        "n_chapters": input("Сколько глав в сказке (1-5): "),
    }

### 4.11. Главная функция

`run_demo` запускает все остальные функции по порядку. Режим можно выбрать вопросом или параметром:

| Режим | Что делает |
|---|---|
| `"1"` | вы вводите героев, место, цель и выбор ребёнка |
| `"2"` | берёт пример из `test_sample.jsonl` и сравнивает с настоящей главой |
| `"3"` | вы загружаете свой файл jsonl с запросами (Colab) |
| `"4"` | проверка качества: 10 тестовых подсказок, доля названных героев у модели и у настоящих глав |

In [ ]:
def run_demo(mode=None, request=None, index=0, adapter_dir=ADAPTER_DIR, sample_path=SAMPLE_PATH,
             history_path=HISTORY_PATH, base_name=BASE_MODEL, preload=True):
    # Главная функция: цепочка загрузка -> очистка -> предобработка -> loader -> модель -> генерация
    # -> постобработка -> проверка героев. request: готовый запрос (словарь), чтобы не вводить вручную.
    plot_history(history_path)
    if mode is None and request is None:
        mode = input("Режим: 1 — ввести свою сказку, 2 — пример из теста, 3 — загрузить файл, 4 — проверка качества [1]: ").strip() or "1"
    model, tok = get_model(base_name, adapter_dir, preload)

    if mode == "4":
        res = evaluate_sample(model, tok, load_examples(sample_path), n=10)
        print(f"Доля названных героев на {res['n']} тестовых подсказках: модель {res['model']:.0%}, "
              f"настоящие главы {res['reference']:.0%}")
        return res

    reference = None
    if request is not None:
        raw = request
    elif mode == "2":
        examples = load_examples(sample_path)
        raw = examples[index % len(examples)]
        reference = raw.get("completion")
    elif mode == "3":
        raw = upload_examples()[0]
    else:
        raw = ask_request()

    clean = clean_request(raw)
    prompt = preprocess_request(clean)
    loader = make_loader([prompt], tok)
    text = postprocess_text(generate_chapters(model, tok, loader)[0])
    named, missing = check_heroes(clean["heroes"], text)
    show_result(clean, text, named, missing)
    if reference:
        print("\nНастоящая глава из датасета (для сравнения), начало:")
        print(reference.replace("</s>", "")[:600], "...")
    return text

## 5. Запуск

Запустите ячейку ниже и отвечайте на вопросы. Если нужно без вопросов, можно, например, так: `run_demo(mode="2", index=3)` (пример №3 из теста) или `run_demo(request={"heroes": "Маша, Медведь", "place": "лесная поляна", "goal": "найти дорогу домой", "choice": "идти за светлячками"})`.

In [ ]:
_ = run_demo()   # результат печатается функцией; «_ =» не даёт Colab повторить текст ещё раз